# Tipps zu Aufgabe 3b) des Arbeitsblattes 6

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tipp 1 </h2>
    <details>
    <summary>Hier klicken!</summary>
    
  Um die Diffusion zu implementieren, kannst du gern in das vorhergehenden Arbeitsblatt zum [Grover-Algorithmus](../worksheets/AB5_Grover.ipynb) schauen.
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tipp 2 </h2>
    <details>
    <summary>Hier klicken!</summary>
    
  Deine Implementierung für die Diffusion kann beispielsweise so aussehen:

  ```python
  def diffusion(circuit, suche, phase):
    """Diffusion zur Amplituden-Verstärkung"""
    circuit.h(suche)
    circuit.x(suche)
    circuit.h(phase)
    circuit.mcx(suche, phase)
    circuit.h(phase)
    circuit.x(suche)
    circuit.h(suche)
  ```
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tipp 3 </h2>
    <details>
    <summary>Hier klicken!</summary>
    
  Um das Such- und Phasenregister zu initialisieren, brauchst du Hadamard-Gates über alle Qubits der Register.

  ```python
  circuit.h(suche)
  circuit.h(phase)
  ```
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tipp 4</h2>
    <details>
    <summary>Hier klicken!</summary>
    
  Um das Input-Register zu initialisieren, kannst du die Funktion `init_input` aus der vorhergehenden Aufgabe nutzen.

  ```python
  init_input(start_feld, grover_circuit, input_register)
  ```
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tipp 5 </h2>
    <details>
    <summary>Hier klicken!</summary>
    
  Für die Messung müsst ihr nun überlegen, wo genau eure Lösung kodiert ist. In Frage kommen das Such- und das Input-Register. Auf dem Input-Register werden die unterschiedlichen Kombinationen geschrieben und ihr testet mit dem Orakel, ob alles null ist. Auf dem Such-Register ist abgebildet, welche Lichter gedrückt werden.

  Könnt ihr raten, welches ihr braucht? Ansonsten probiert gern auch einfach mal beide nacheinander aus.
</details>
</div>

<div class="alert alert-info">
  <h2><i class="fas fa-life-ring" style="font-size:36px"></i> &nbsp; Tipp 6 </h2>
    <details>
    <summary>Hier klicken!</summary>
    
  Ihr habt alles korrekt implementiert und könnt die Lösung trotzdem nicht ablesen? Versucht die Anzahl der Iterationen zu erhöhen. Mit nur einer Iteration werdet ihr vermutlich noch nicht die richtige Lösung sehen, da die Amplitude nicht hoch genug ist.

  ```python
  grover_lights_out(grover_circuit, input_register, such_register, phase_register, 3)
  ```

  Wisst ihr, mit welcher Anzahl an Iterationen ihr das optimale Ergebnis bekommt? Auch das könnt ihr einfach ausprobieren, wenn ihr wollt.
</details>
</div>

<div class="alert alert-success">
  <h2><i class="fas fa-info" style="font-size:36px"></i> &nbsp;  Musterlösung </h2>
  <details>
  <summary>Hier klicken!</summary>
  
  ```python
  from qiskit_aer import AerSimulator
  import math
  from qiskit import ClassicalRegister

  start_feld = [[0, 1, 0],
                [1, 1, 1],
                [0, 1, 0]]

  input_register = QuantumRegister(len(start_feld)**2, "data")
  such_register = QuantumRegister(len(start_feld)**2, "oracle")
  phase_register = QuantumRegister(1, "phase")
  classical_register = ClassicalRegister(len(start_feld)**2, "output")

  grover_circuit = QuantumCircuit(input_register, such_register, phase_register, classical_register)

  def diffusion(circuit, suche, phase):
      """Diffusion zur Amplituden-Verstärkung"""
      circuit.h(suche)
      circuit.x(suche)
      circuit.h(phase)
      circuit.mcx(suche, phase)
      circuit.h(phase)
      circuit.x(suche)
      circuit.h(suche)

  def grover_lights_out(circuit, data, suche, phase, iterationen=1):
      """Löst Lights-out mit Grover"""
      
      circuit.h(suche)
      circuit.h(phase)
      
      for _ in range(iterationen):
          # Orakel
          lights_out_oracle(circuit, data, suche, phase)
          # Diffusion
          diffusion(circuit, suche, phase)
      

  init_input(start_feld, grover_circuit, input_register)

  grover_lights_out(grover_circuit, input_register, such_register, phase_register, 3)

  grover_circuit.measure(such_register, classical_register)

  grover_circuit.reverse_bits()  # Umdrehen der Codierung zum korrekten Ablesen

  print("Grover für Lights-out:")
  print(grover_circuit.draw())

  # Simulation
  simulator = AerSimulator()
  job = simulator.run(grover_circuit, shots=100)
  result = job.result()
  counts = result.get_counts()
  print(counts)

  print("\nTop 3 Ergebnisse:")
  for state, count in sorted(counts.items(), key=lambda x: -x[1])[:3]:
      print(f"  {state}: {count}%")
  ```
  </details>
</div>